# RED NEURONAL RECURRENTE PARA SERIES TEMPORALES CON PYTORCH

## CREAMOS DATASET DE EJEMPLO

In [1]:
import numpy as np

# Datos de ejemplo (serie temporal simple)
data = np.array([10, 20, 30, 40, 50, 60, 70, 80])
data

array([10, 20, 30, 40, 50, 60, 70, 80])

In [2]:
# Función para crear secuencias
def create_sequences(data, seq_length=3):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])  # Secuencia de entrada (X)
        y.append(data[i+seq_length])    # Valor a predecir (y)
    return np.array(X), np.array(y)

In [3]:
# Crear secuencias con longitud 3
seq_length = 3
X, y = create_sequences(data, seq_length)
# Mostrar los resultados
print("X (Secuencias de entrada):")
print(X)
print("\ny (Valores esperados):")
print(y)

X (Secuencias de entrada):
[[10 20 30]
 [20 30 40]
 [30 40 50]
 [40 50 60]
 [50 60 70]]

y (Valores esperados):
[40 50 60 70 80]


In [4]:
# Redimensionar X para que tenga la forma adecuada para LSTM: (samples, timesteps, features)
X = X.reshape((X.shape[0], X.shape[1], 1))  # Agregar dimensión de características
X

array([[[10],
        [20],
        [30]],

       [[20],
        [30],
        [40]],

       [[30],
        [40],
        [50]],

       [[40],
        [50],
        [60]],

       [[50],
        [60],
        [70]]])

# CREAMOS NUESTRA RED NEURONAL RECURRENTE CON PYTORCH

# CONVERTIR NUESTRA SERIE TEMPORAL A TENSORES DE PYTORCH

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

In [8]:
X_tensor = torch.tensor(X, dtype=torch.float32)
type(X_tensor)

torch.Tensor

In [11]:
y_tensor = torch.tensor(y,dtype=torch.float32)

## CREAMOS NUESTRA RED NEURONAL RECURRENTE SIMPLE

In [9]:
class SimpleRNN(nn.Module):
  def __init__(self,input_size,hidden_size,output_size):
    super(SimpleRNN,self).__init__()
    self.hidden_size = hidden_size
    self.rnn = nn.RNN(input_size,hidden_size,batch_first=True)
    self.fc = nn.Linear(hidden_size,output_size)

  def forward(self,x,hidden):
    out,hidden = self.rnn(x,hidden)
    out = self.fc(out[:,-1,:])
    return out,hidden

  def init_hidden(self,batch_size):
    return torch.zeros(1,batch_size,self.hidden_size)

# CONFIGURAMOS NUESTRA RED NEURONAL RNN

In [10]:
input_size = 1
hidden_size = 5
output_size = 1
rnn = SimpleRNN(input_size,hidden_size,output_size)
criterion = nn.MSELoss()
optimizer = optim.SGD(rnn.parameters(),lr=0.01)

## ENTRENAMOS NUESTRA RED RNN

In [16]:
epochs = 100
for epoch in range(epochs):
  hidden = rnn.init_hidden(5)
  optimizer.zero_grad()
  output,hidden = rnn(X_tensor,hidden)
  loss = criterion(output,y_tensor.view(-1, 1))
  loss.backward()
  optimizer.step()
  print(f'Epoch {epoch}, Loss : {loss.item()}')

Epoch 0, Loss : 201.7040557861328
Epoch 1, Loss : 201.3162841796875
Epoch 2, Loss : 201.01431274414062
Epoch 3, Loss : 200.7769317626953
Epoch 4, Loss : 200.58200073242188
Epoch 5, Loss : 200.36317443847656
Epoch 6, Loss : 189.61549377441406
Epoch 7, Loss : 629.003662109375
Epoch 8, Loss : 532.2203369140625
Epoch 9, Loss : 457.2713928222656
Epoch 10, Loss : 399.230712890625
Epoch 11, Loss : 354.2840576171875
Epoch 12, Loss : 319.4774475097656
Epoch 13, Loss : 292.52325439453125
Epoch 14, Loss : 271.6498107910156
Epoch 15, Loss : 255.4853973388672
Epoch 16, Loss : 242.9677276611328
Epoch 17, Loss : 233.27401733398438
Epoch 18, Loss : 225.76724243164062
Epoch 19, Loss : 219.9539337158203
Epoch 20, Loss : 215.4521026611328
Epoch 21, Loss : 211.96597290039062
Epoch 22, Loss : 209.2661590576172
Epoch 23, Loss : 207.175537109375
Epoch 24, Loss : 205.5565643310547
Epoch 25, Loss : 204.30270385742188
Epoch 26, Loss : 203.33180236816406
Epoch 27, Loss : 202.57997131347656
Epoch 28, Loss : 201.9

## EVALUAMOS EL MODELO ENTRENADO

In [17]:
with torch.no_grad():
    hidden = rnn.init_hidden(5)
    predicted, hidden = rnn(X_tensor, hidden)
    print(predicted)
print("predicted: ", predicted)

tensor([[54.0398],
        [54.0601],
        [54.0607],
        [54.0607],
        [54.0607]])
predicted:  tensor([[54.0398],
        [54.0601],
        [54.0607],
        [54.0607],
        [54.0607]])
